<a href="https://colab.research.google.com/github/boyerb/Investments/blob/master/Ex08-AlphaBeta_Alpha_Vantage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Investments: Theory, Fundamental Analysis, and Data Driven Analytics**, Bates, Boyer, and Fletcher

# Example Chapter 8: Estimating Alpha Using Data from Alpha Vantage API  
  
In this example we estimate the alpha and beta of Johnson and Johnson (`ticker:JNJ`) relative to the market portfolio along with 95% confidence intervals. In this example, data for Johnson and Johnson returns come from the Alpha Vantage API. Data for the market and short-term T-bill returns come from [Ken French's website](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html). An introduction to the Alpha Vantage API is given in [EX07-Alpha_Vantage.ipynb](https://colab.research.google.com/github/boyerb/Investments/blob/master/Ex07-Alpha_Vantage.ipynb).  

We install and import the current `farms` package. Its Alpha Vantage loader reads the API key securely, downloads adjusted monthly returns, and adds the market excess return and risk-free rate used in the regression.

In [ ]:
# Install the current farms package and import the tools used below.
%pip install -q --upgrade farms

import farms as fm
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', 20)


`farms.get_alpha_vantage_api_key()` reads the key from Google Colab Secrets or the `ALPHAVANTAGE_API_KEY` environment variable. The loader handles the Alpha Vantage request and returns adjusted monthly returns together with the market excess return and risk-free rate.

In [ ]:
# Load adjusted monthly returns for JNJ and merge market factors.
api_key = fm.get_alpha_vantage_api_key()
df = fm.load_alpha_vantage(
    symbol="JNJ",
    api_key=api_key,
    frequency="monthly",
    field="returns",
    start_date="2010-01",
    end_date="2024-12",
    include_factors="market",
)

df = df.dropna(subset=["Return", "mkt-rf", "rf"]).copy()
df["Ret-RF"] = df["Return"] - df["rf"]
df


### Load Returns for the Market and Short-Term T-Bills
This function retrieves monthly data for the Fama–French Five-Factor model from [Ken French's website](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html). From the resulting DataFrame, we will extract the market’s excess return `Mkt-RF` and the return on short-term T-bills `RF`.


In [ ]:
# The farms loader already added the market excess return and risk-free rate.
df[["Return", "mkt-rf", "rf", "Ret-RF"]]


### Merge Two Data Sources
In this code we merge the data from Alpha Vantage and Ken French's website together. We then create the excess return for Johnson & Johnson.

In [ ]:
merged = df[["Return", "mkt-rf", "rf", "Ret-RF"]].dropna()
merged


### Estimate Alpha, Beta, and 95% Confidence Intervals
Here we estimate the alpha and beta of Johnson & Johnson relative to the market portfolio along with the 95% confidence intervals.

In [ ]:
results = fm.run_ols(merged["mkt-rf"], merged["Ret-RF"])
results
